In [ ]:
# Import libraries
import prefect
from prefect import flow, task
import numpy as np
import pandas as pd

print(f"Prefect version: {prefect.__version__}")
print(f"API URL: {prefect.settings.PREFECT_API_URL.value()}")

## Run the Bundled Training Flow

Let's run the bundled ML training pipeline that's included in the cluster.

In [ ]:
# Load and run the bundled training flow
import sys
sys.path.insert(0, '/home/jovyan/bundled-flows')

from train_model import ml_training_pipeline

# Run with custom parameters
result = ml_training_pipeline(
    n_samples=500,
    n_features=15,
    n_estimators=50,
    max_depth=5
)

print(f"\nTraining completed with accuracy: {result['accuracy']:.4f}")

## Create a Simple Flow

Let's create a simple Prefect flow to demonstrate the basics.

In [ ]:
@task(name="generate_data")
def generate_data(n=100):
    """Generate sample data"""
    data = np.random.randn(n, 3)
    df = pd.DataFrame(data, columns=['feature_1', 'feature_2', 'feature_3'])
    print(f"Generated {len(df)} samples")
    return df

@task(name="analyze_data")
def analyze_data(df):
    """Analyze the data"""
    stats = df.describe()
    print("\nData Statistics:")
    print(stats)
    return stats

@flow(name="simple-analysis-flow")
def simple_analysis_flow(n_samples=100):
    """A simple data analysis flow"""
    data = generate_data(n_samples)
    stats = analyze_data(data)
    return stats

# Run the flow
result = simple_analysis_flow(n_samples=200)

## Deploy a Flow to Prefect

Deploy your flow to run on the Prefect worker pool.

In [ ]:
# Deploy the flow to the work pool
# Note: This requires the flow code to be accessible to workers

# simple_analysis_flow.deploy(
#     name="simple-analysis",
#     work_pool_name="default-pool",
#     cron="0 */12 * * *",  # Run every 12 hours
#     description="Simple data analysis pipeline"
# )

print("To deploy flows from JupyterLab, you'll need to:")
print("1. Save your flow code to a Python file")
print("2. Make it accessible to workers (via ConfigMap or container image)")
print("3. Use the deploy() method as shown above")

## Check Prefect Server Status

In [ ]:
from prefect import get_client

async def check_server_status():
    async with get_client() as client:
        # Get work pools
        work_pools = await client.read_work_pools()
        print("Available Work Pools:")
        for pool in work_pools:
            print(f"  - {pool.name} ({pool.type})")
        
        # Get recent flow runs
        flow_runs = await client.read_flow_runs(limit=5)
        print(f"\nRecent Flow Runs: {len(flow_runs)}")
        for run in flow_runs:
            print(f"  - {run.name}: {run.state.type}")

# Run the async function
import asyncio
await check_server_status()

## Next Steps

1. Explore the bundled flows in `/home/jovyan/bundled-flows/`
2. Create your own ML workflows using Prefect tasks and flows
3. View flow runs in the Prefect UI at http://localhost:4200
4. Save your notebooks to the `work/` directory for persistence

## Useful Links

- [Prefect Documentation](https://docs.prefect.io/3.0/)
- [Prefect UI](http://localhost:4200) (via port-forward)
- [Bundled Training Script](/home/jovyan/bundled-flows/train_model.py)